In [18]:
import numpy as np

**Checking the SetCoverPy:**
To see how it works on its own test data

In [19]:
pip install SetCoverPy

In [20]:
import os
import urllib.request
import tarfile

# Set the URL for the test data archive
url = "https://s3.us-east-2.amazonaws.com/setcoverproblem/BeasleyOR/scp_BeasleyOR_numpyformat.tar.gz"

# Set the destination directory for extraction
destination_dir = "BeasleyOR"

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Download the test data archive
tar_gz_file = os.path.join(destination_dir, "scp_BeasleyOR_numpyformat.tar.gz")
urllib.request.urlretrieve(url, tar_gz_file)

# Extract the contents of the archive
with tarfile.open(tar_gz_file, "r:gz") as tar:
    tar.extractall(destination_dir)

In [21]:
from SetCoverPy import setcover

# Load the test data
# Update the paths based on the correct directory structure
a_matrix_path = "/content/BeasleyOR/scp_BeasleyOR_numpyformat/scpb4_matrix.npy"
cost_path = "/content/BeasleyOR/scp_BeasleyOR_numpyformat/scpb4_cost.npy"

# Load the test data
a_matrix = np.load(a_matrix_path)
cost = np.load(cost_path)

# Instantiate SetCover object
g = setcover.SetCover(a_matrix, cost)

# Run the solver
solution, time_used = g.SolveSCP()

This Best solution: UB=80.0, LB=69.83089855291234, UB1=84.0, LB1=69.93499637563609
Current Best Solution: UB=80, LB=69.83089855291234, change=12.71137680885957% @ niters=0
This Best solution: UB=80.0, LB=70.45816056786438, UB1=87.0, LB1=70.57099981643915
Current Best Solution: UB=79, LB=70.66857121240805, change=10.546112389356894% @ niters=5
This Best solution: UB=85.0, LB=-116.56551129971193, UB1=85.0, LB1=-116.56551129971193
Current Best Solution: UB=79, LB=70.66857121240805, change=10.546112389356894% @ niters=10
This Best solution: UB=79.0, LB=70.20193543115266, UB1=87.0, LB1=70.35632297120834
Current Best Solution: UB=79, LB=70.20193543115266, change=11.136790593477643% @ niters=15
Iteration in re-initialization reaches maximum number = 20
Current Best Solution: UB=79, LB=70.70992701656483, change=10.493763270171097% @ niters=20
Final Best solution: 79
Took 2.989 minutes to reach current solution.


In [22]:
# Print the loaded test data
print("Binary Relationship Matrix (a_matrix):")
print(a_matrix)

print("\nCost of Columns (cost):")
print(cost)

Binary Relationship Matrix (a_matrix):
[[ True False False ... False False False]
 [False False False ... False False False]
 [ True False False ... False False False]
 ...
 [ True False False ... False False False]
 [False False False ... False False False]
 [ True False False ... False False False]]

Cost of Columns (cost):
[  1   1   1 ... 100 100 100]


In [23]:
# Print the solution and total cost
print("Solution (Selected Columns):", solution)
print("Total Cost of the Set Cover:", g.total_cost)

Solution (Selected Columns): 79
Total Cost of the Set Cover: 79


In [24]:
print("Solution (g.s):", g.s)
print("Total Cost of the Set Cover:", g.total_cost)

Solution (g.s): [ True  True  True ... False False False]
Total Cost of the Set Cover: 79


**New Way to build Matrix**

In [25]:
def generate_binary_matrices2(num_matrices, max_universe_size):
    binary_matrices = []

    for _ in range(num_matrices):
        # Generate a random number of rows and columns for each matrix
        num_rows = np.random.randint(3, max_universe_size + 1) #min universe size is 3
        num_cols = np.random.randint(1, max_universe_size + 1)

         # Create a random binary matrix with at least one True value in each row
        binary_matrix = np.zeros((num_rows, num_cols), dtype=bool)
        for i in range(num_rows):
            # Ensure at least one True value in each row
            binary_matrix[i, np.random.randint(0, num_cols)] = True

            # Ensure at least one True value in each column
        for j in range(num_cols):
            binary_matrix[np.random.randint(0, num_rows), j] = True

        binary_matrices.append(binary_matrix)

    return binary_matrices

In [26]:
# define parameters
num_matrices = 10
max_universe_size = 6

# Generate binary matrices
generated_matrices = generate_binary_matrices2(num_matrices, max_universe_size)

In [27]:
# Print the generated matrices
for i, matrix in enumerate(generated_matrices):
    print(f"Generated Binary Matrix {i + 1}:")
    print(matrix)
    print()

Generated Binary Matrix 1:
[[ True False False  True False]
 [False  True False False False]
 [False  True  True False  True]
 [False False False  True  True]]

Generated Binary Matrix 2:
[[ True False False  True  True]
 [ True False  True False False]
 [False  True False False False]]

Generated Binary Matrix 3:
[[ True  True]
 [False  True]
 [False  True]]

Generated Binary Matrix 4:
[[ True  True False False False]
 [False False  True  True False]
 [False  True False False  True]]

Generated Binary Matrix 5:
[[ True False False  True  True]
 [False False False  True False]
 [ True  True  True False False]]

Generated Binary Matrix 6:
[[ True False False  True]
 [False  True False  True]
 [False False  True False]
 [ True False False False]
 [ True False False False]
 [ True False  True False]]

Generated Binary Matrix 7:
[[ True  True]
 [False  True]
 [False  True]
 [False  True]]

Generated Binary Matrix 8:
[[False  True False False  True False]
 [False False False  True False Fal

In [28]:
# Function to convert a binary matrix to SetCoverPy input format
def convert_to_setcover_input(binary_matrix):
    return binary_matrix, np.zeros(binary_matrix.shape[1], dtype=float)

# List to store SetCoverPy solutions
setcover_solutions = []

# Iterate over the matrices and find set cover solutions
for i, generated_matrix in enumerate(generated_matrices):
    # Convert binary matrix to SetCoverPy input format
    setcover_input = convert_to_setcover_input(generated_matrix)

    # Instantiate SetCover object
    g = setcover.SetCover(*setcover_input)

    # Run the solver
    solution, time_used = g.SolveSCP()

    # Save the solution to the list
    setcover_solutions.append(g.s)

    # Print the solution and total cost
    print(f"Set Cover Solution for Matrix {i + 1}:")
    print("Solution (g.s):", g.s)
    print("Total Cost of the Set Cover:", g.total_cost)
    print()

# Now 'setcover_solutions' contains the binary vectors of solutions for each matrix

This Best solution: UB=0.0, LB=-0.016541244549457074, UB1=0.0, LB1=-0.016541244549457074
Current Best Solution: UB=0.0, LB=-0.016541244549457074, change=inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=2
Final Best solution: 0.0
Took 0.004 minutes to reach current solution.
Set Cover Solution for Matrix 1:
Solution (g.s): [ True  True  True  True False]
Total Cost of the Set Cover: 0.0



/usr/local/lib/python3.10/dist-packages/SetCoverPy/setcover.py:484: RuntimeWarning: divide by zero encountered in double_scalars
  GAP = (UB-LB)/np.fabs(UB)


This Best solution: UB=0.0, LB=-0.030835141873653195, UB1=0.0, LB1=-0.030835141873653195
Current Best Solution: UB=0.0, LB=-0.030835141873653195, change=inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=2
Final Best solution: 0.0
Took 0.003 minutes to reach current solution.
Set Cover Solution for Matrix 2:
Solution (g.s): [ True  True False False False]
Total Cost of the Set Cover: 0.0



/usr/local/lib/python3.10/dist-packages/SetCoverPy/setcover.py:364: RuntimeWarning: invalid value encountered in double_scalars
  u_temp = u_sequence[:,i]+self._stepsize*(UB - Lu_sequence[i])/s_u_norm*s_u


This Best solution: UB=0.0, LB=1e-05, UB1=0.0, LB1=1e-05
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=1
Final Best solution: 0.0
Took 0.002 minutes to reach current solution.
Set Cover Solution for Matrix 3:
Solution (g.s): [False  True]
Total Cost of the Set Cover: 0.0

This Best solution: UB=0.0, LB=1e-05, UB1=0.0, LB1=1e-05
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=1
Final Best solution: 0.0
Took 0.003 minutes to reach current solution.
Set Cover Solution for Matrix 4:
Solution (g.s): [ True  True  True False False]
Total Cost of the Set Cover: 0.0

This Best solution: UB=0.0, LB=-0.0007283730862420459, UB1=0.0, LB1=-0.0007283730862420459
Current Best Solution: UB=0.0, LB=-0.0007283730862420459, change=inf% @ niters=0
Current Best Solution: UB=0.0, LB=1e-05, change=-inf% @ niters=2
Final Best solution: 0.0
Took 0.004

In [29]:
# Print the setcover_solutions list separately
for i, solution in enumerate(setcover_solutions):
    print(f"Set Cover Solution for Matrix {i + 1}:")
    print("Solution (g.s):", solution)
    print()

Set Cover Solution for Matrix 1:
Solution (g.s): [ True  True  True  True False]

Set Cover Solution for Matrix 2:
Solution (g.s): [ True  True False False False]

Set Cover Solution for Matrix 3:
Solution (g.s): [False  True]

Set Cover Solution for Matrix 4:
Solution (g.s): [ True  True  True False False]

Set Cover Solution for Matrix 5:
Solution (g.s): [ True False False  True False]

Set Cover Solution for Matrix 6:
Solution (g.s): [ True  True  True False]

Set Cover Solution for Matrix 7:
Solution (g.s): [False  True]

Set Cover Solution for Matrix 8:
Solution (g.s): [False False False  True  True False]

Set Cover Solution for Matrix 9:
Solution (g.s): [ True  True  True False False]

Set Cover Solution for Matrix 10:
Solution (g.s): [ True]



**Converting everything into sequence:**

In [30]:
def convert_to_sequences(binary_matrices, solutions):
    sequences = []

    for binary_matrix, solution in zip(binary_matrices, solutions):
        # Convert binary matrix to a sequence of subsets
        input_sequence = [list(np.where(column)[0]) for column in binary_matrix.T]

        # Convert solution to a sequence of integers
        #label_sequence = list(np.where(solution)[0])
        # Convert solution to a sequence of integers starting from 1
        label_sequence = [index + 1 for index in np.where(solution)[0]]

        sequences.append((input_sequence, label_sequence))

    return sequences

In [31]:
# (Assuming 'generated_matrices' contains the binary matrices and 'setcover_solutions' contains the SetCoverPy solutions)
# Input is the list of matrices, output is the list of integers

# Convert binary matrices and solutions to sequences
sequences = convert_to_sequences(generated_matrices, setcover_solutions)

# Print the converted sequences
for i, (input_sequence, label_sequence) in enumerate(sequences):
    print(f"Sequence {i + 1} - Input:")
    print(input_sequence)
    print("Label:")
    print(label_sequence)
    print()

Sequence 1 - Input:
[[0], [1, 2], [2], [0, 3], [2, 3]]
Label:
[1, 2, 3, 4]

Sequence 2 - Input:
[[0, 1], [2], [1], [0], [0]]
Label:
[1, 2]

Sequence 3 - Input:
[[0], [0, 1, 2]]
Label:
[2]

Sequence 4 - Input:
[[0], [0, 2], [1], [1], [2]]
Label:
[1, 2, 3]

Sequence 5 - Input:
[[0, 2], [2], [2], [0, 1], [0]]
Label:
[1, 4]

Sequence 6 - Input:
[[0, 3, 4, 5], [1], [2, 5], [0, 1]]
Label:
[1, 2, 3]

Sequence 7 - Input:
[[0], [0, 1, 2, 3]]
Label:
[2]

Sequence 8 - Input:
[[2], [0, 3], [3], [1, 3], [0, 2, 4, 5], [4]]
Label:
[4, 5]

Sequence 9 - Input:
[[0, 3, 5], [2, 4], [1, 4], [4], [0]]
Label:
[1, 2, 3]

Sequence 10 - Input:
[[0, 1, 2, 3]]
Label:
[1]

